In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "figure.constrained_layout.use": True,
})

ALGO_SHORT = {
    "ModifiedLogPartitionVarianceGFlowNet": "ModLPV",
    "LogPartitionVarianceGFlowNet": "LPV",
    "TBGFlowNet": "TB",
    "ModifiedTBGFlowNet": "ModTB",
}
ALGO_ORDER = ["TB", "ModTB", "LPV", "ModLPV"]
COLORS = {"TB": "C0", "ModTB": "C1", "LPV": "C2", "ModLPV": "C3"}

sweep_dir = Path("sweep_results")

def _load_config(path):
    """Load first JSON object from a file (some have a trailing brace)."""
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return json.JSONDecoder().raw_decode(path.read_text())[0]

# --- Load summary ---
summary = pd.read_csv(sweep_dir / "sweep_summary.csv")
summary["algo"] = summary["algorithm"].map(ALGO_SHORT)

# --- Load per-run timeseries ---
frames = []
for csv_path in sorted(sweep_dir.glob("*_results.csv")):
    run_id = csv_path.name.split("_results")[0]
    df = pd.read_csv(csv_path, low_memory=False)
    cfg_path = sweep_dir / f"{run_id}_config.json"
    if cfg_path.exists():
        cfg = _load_config(cfg_path).get("config_snapshot", {}).get("config", {})
        for k in ["lr", "beta2", "grad_clip_max_norm"]:
            if k in cfg and k not in df.columns:
                df[k] = cfg[k]
    df["run_id"] = run_id
    frames.append(df)

ts = pd.concat(frames, ignore_index=True)
ts["algo"] = ts["algorithm"].map(ALGO_SHORT)
ts["l1_finite"] = ts["l1_dist"].replace([np.inf, -np.inf], np.nan)

# --- Convergence metric: iteration to find all 16 modes ---
def _first_full(g):
    full = g[g["n_modes_found"] >= 16]
    return full["iteration"].min() if len(full) else np.nan

convergence = (
    ts.groupby(["environment", "algo", "run_id", "seed"])
    .apply(_first_full, include_groups=False)
    .reset_index(name="iter_16")
)

# --- Best run per (algo, env): fastest to find all 16 modes ---
best_conv = convergence.sort_values("iter_16").groupby(["environment", "algo"]).first().reset_index()
best_conv_keys = set(zip(best_conv["environment"], best_conv["algo"], best_conv["run_id"], best_conv["seed"]))
ts_best = ts[ts[["environment", "algo", "run_id", "seed"]].apply(tuple, axis=1).isin(best_conv_keys)].copy()

env_list = sorted(ts["environment"].unique())

print(f"Summary: {summary.shape[0]} configs, {summary['algo'].nunique()} algorithms, "
      f"{summary['environment'].nunique()} environments")
print(f"Timeseries: {ts.shape[0]:,} rows from {ts['run_id'].nunique()} run files")
print(f"Best runs selected: {len(best_conv_keys)} (by fastest mode discovery)")

## Convergence speed: iterations to find all 16 modes

The primary metric is how quickly each method discovers all modes of the reward distribution. For each (algorithm, environment) pair, we report statistics across all hyperparameter configurations.

In [ ]:
conv_table = convergence.groupby(["algo", "environment"])["iter_16"].agg(
    best="min", median="median", pct_converged=lambda x: f"{x.notna().mean():.0%}"
).unstack("environment")

print("Iterations to discover all 16 modes (lower = faster convergence):")
conv_table["best"].style.format("{:.0f}", na_rep="never").background_gradient(
    axis=None, cmap="RdYlGn_r"
)

In [ ]:
print("Fraction of runs that ever found all 16 modes:")
conv_table["pct_converged"]

## Mode discovery over training (best run per algorithm)

For each (algorithm, environment), we plot the single best hyperparameter configuration — the run that reached all 16 modes fastest.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, env in zip(axes.flat, env_list):
    for algo in ALGO_ORDER:
        ad = ts_best[(ts_best["environment"] == env) & (ts_best["algo"] == algo)]
        if ad.empty:
            continue
        ax.plot(ad["iteration"], ad["n_modes_found"], label=algo, color=COLORS[algo], linewidth=1.5)
    ax.axhline(16, ls="--", color="gray", alpha=0.4, label="16 (all modes)")
    ax.set_title(env)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Modes found")
    ax.set_ylim(-0.5, 17.5)
    ax.legend(loc="lower right")

fig.suptitle("Mode discovery — best run per algorithm", fontsize=13)
plt.show()

## Loss over training (best run per algorithm)

Loss values are not directly comparable across methods (different constructions), but shown for reference alongside the same best runs.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, env in zip(axes.flat, env_list):
    for algo in ALGO_ORDER:
        ad = ts_best[(ts_best["environment"] == env) & (ts_best["algo"] == algo)]
        if ad.empty:
            continue
        ax.plot(ad["iteration"], ad["loss"].clip(upper=200),
                label=algo, color=COLORS[algo], linewidth=1.2, alpha=0.8)
    ax.set_title(env)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.set_yscale("symlog", linthresh=0.1)
    ax.legend(loc="upper right")

fig.suptitle("Training loss — best run per algorithm (not cross-comparable)", fontsize=13)
plt.show()

## Hyperparameter sensitivity

For each algorithm, how does the best-case convergence speed (iterations to 16 modes) vary with each hyperparameter? Each point = best run at that HP value (min across other HP settings).

In [ ]:
# Merge convergence info with HP columns from summary
conv_with_hp = convergence.merge(
    summary[["algo", "environment", "lr", "beta2", "grad_clip_max_norm", "final_l1"]],
    on=["algo", "environment"],
    how="left",
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, hp in zip(axes, ["lr", "beta2", "grad_clip_max_norm"]):
    for algo in ALGO_ORDER:
        sub = conv_with_hp[(conv_with_hp["algo"] == algo) & conv_with_hp["iter_16"].notna()]
        if sub.empty or hp not in sub.columns or sub[hp].isna().all():
            continue
        grouped = sub.groupby(hp)["iter_16"].min()
        ax.plot(grouped.index, grouped.values, "o-", label=algo, color=COLORS[algo], markersize=5)
    ax.set_xlabel(hp)
    ax.set_ylabel("Best iterations to 16 modes")
    if hp == "lr":
        ax.set_xscale("log")
    ax.legend(fontsize=7)

fig.suptitle("Hyperparameter sensitivity (best convergence speed)", fontsize=13)
plt.show()

## Best hyperparameters per algorithm x environment

In [ ]:
# Show best run details: which HP config converged fastest?
best_detail = best_conv.merge(
    ts.groupby(["environment", "algo", "run_id", "seed"])[["lr", "beta2", "grad_clip_max_norm"]].first().reset_index(),
    on=["environment", "algo", "run_id", "seed"],
    how="left",
)
best_detail = best_detail[["algo", "environment", "iter_16", "lr", "beta2", "grad_clip_max_norm", "run_id"]].copy()
best_detail = best_detail.sort_values(["environment", "iter_16"])
best_detail.style.format({"iter_16": "{:.0f}", "lr": "{:.0e}"}, na_rep="never").background_gradient(
    subset=["iter_16"], cmap="RdYlGn_r"
)

## L1 distance over training (best runs)

**Bug note:** The L1 values in these results were computed by `torchgfn`'s `validate()` using `.mean()` instead of `.sum()`, dividing the true L1 distance by `n_terminating_states` (~331k). This has been fixed locally in `lir.gflownet.tb_normalize.validate`. Future runs will show meaningful L1 convergence curves; the data below is included for completeness but the absolute values are not interpretable.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, env in zip(axes.flat, env_list):
    for algo in ALGO_ORDER:
        ad = ts_best[(ts_best["environment"] == env) & (ts_best["algo"] == algo)].copy()
        if ad.empty:
            continue
        # Filter to finite L1 only
        ad = ad[ad["l1_finite"].notna()]
        if ad.empty:
            continue
        ax.plot(ad["iteration"], ad["l1_finite"], label=algo, color=COLORS[algo], linewidth=1.2)
    ax.set_title(env)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("L1 distance (buggy .mean(), not .sum())")
    ax.set_yscale("log")
    ax.legend(loc="upper right")

fig.suptitle("L1 distance over training — best runs (values ~1/n_states too small)", fontsize=13)
plt.show()

## Training stability: ModLPV vs LPV

ModLPV exhibits loss spikes during training, especially at higher learning rates. The instability is predominantly lr-driven — at lr <= 1e-4, ModLPV is comparably stable to LPV.

In [ ]:
# Compute rolling coefficient of variation (std/mean) over 100-iter windows
def _rolling_cv(g):
    loss = g.sort_values("iteration")["loss"]
    rm = loss.rolling(100, min_periods=20).mean()
    rs = loss.rolling(100, min_periods=20).std()
    return (rs / rm.clip(lower=1e-8)).mean()

stability = (
    ts[ts["algo"].isin(["ModLPV", "LPV"])]
    .groupby(["algo", "environment", "run_id", "seed", "lr"])
    .apply(_rolling_cv, include_groups=False)
    .reset_index(name="cv")
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: CV by learning rate
for ax_i, algo in enumerate(["LPV", "ModLPV"]):
    sub = stability[stability["algo"] == algo]
    bp_data = [sub.loc[sub["lr"] == lr, "cv"].dropna().values for lr in sorted(sub["lr"].unique())]
    bp = axes[0].boxplot(bp_data, positions=np.arange(len(bp_data)) + ax_i * 0.3 - 0.15,
                         widths=0.25, patch_artist=True, showfliers=False,
                         boxprops=dict(facecolor=COLORS[algo], alpha=0.4),
                         medianprops=dict(color="black"))
    axes[0].plot([], [], color=COLORS[algo], label=algo, linewidth=5, alpha=0.4)

lr_labels = [f"{lr:.0e}" for lr in sorted(stability["lr"].unique())]
axes[0].set_xticks(range(len(lr_labels)))
axes[0].set_xticklabels(lr_labels)
axes[0].set_xlabel("Learning rate")
axes[0].set_ylabel("Rolling CV of loss (instability)")
axes[0].legend()
axes[0].set_title("Stability by learning rate")

# Right: best-run loss curves overlaid for ModLPV vs LPV (original env, to see spike pattern)
for algo in ["LPV", "ModLPV"]:
    ad = ts_best[(ts_best["environment"] == "original") & (ts_best["algo"] == algo)]
    if ad.empty:
        continue
    axes[1].plot(ad["iteration"], ad["loss"], label=algo, color=COLORS[algo], linewidth=1, alpha=0.8)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Loss")
axes[1].set_yscale("symlog", linthresh=0.01)
axes[1].set_title("Best-run loss: original env")
axes[1].legend()

fig.suptitle("Training stability comparison", fontsize=13)
plt.show()